In [ ]:
import sys
# sys.path.append('../Bayesflow/')
# import bayesflow as bf
import numpy as np
import numba
import pandas as pd
from numba import njit
import matplotlib.pyplot as plt
import seaborn as sns
import notebook
import pickle
from scipy.stats import skew
import xgboost as xgb

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, StratifiedGroupKFold
# from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score,
    classification_report, confusion_matrix, accuracy_score, brier_score_loss, balanced_accuracy_score,
    precision_recall_curve, f1_score, precision_score, fbeta_score, matthews_corrcoef)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from joblib import dump, Memory
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import matthews_corrcoef

In [ ]:
from imblearn.pipeline import Pipeline as imbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.ensemble import BalancedRandomForestClassifier

In [ ]:
# demo = pd.read_csv("datasets/demog_HC3all_large.csv")
demo = pd.read_csv("datasets/demog_HC3_full_py.csv")
# demo = pd.read_csv("datasets/data_small.csv")

In [ ]:
# remove null values in parameters: short emse
demo = demo[~demo['SHORT_EMSE'].isnull()]
demo = demo.reset_index()

In [ ]:
demo["age_bin"] = pd.qcut(demo["DOC_AGE_HC3"], q=4)
demo["strata"] = (
    demo["INC_DEMENTIA_ALL"].astype(str) + "|" +
    demo["age_bin"].astype(str) + "|" +
    demo["SEX"].astype(str)
)

In [ ]:
demo["y"] = demo["INC_DEMENTIA_ALL"].map({"No Dementia":0, "Dementia":1})
demo["sex"] = demo["SEX"].map({"Men":0, "Women":1})

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, solver="liblinear"), #class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=30, max_depth=7, random_state=42),
    "RBF SVM": SVC(kernel="rbf", probability=True, C=0.1, gamma=1e-3),# class_weight='balanced')
    "Gaussian NB": GaussianNB(var_smoothing=1e-9),
    "XGBoost": xgb.XGBClassifier(objective='binary:logistic', n_estimators=500, random_state=42, 
                                 max_depth=4, min_child_weight=5, gamma=1, scale_pos_weight=1, subsample=0.8, colsample_bytree=0.8,
                                 learning_rate=0.3, eval_metric='aucpr') #early_stopping_rounds=200,
}

# RT model

In [ ]:
demo_rt = demo
cols_rt = ["DOC_AGE_HC3", "EDLEVEL09", "SHORT_EMSE", "MI3", "CVA3", "DM3", "sex", "marital_st", "rt_q1", "rt_q2","rt_q3"] # "DEPRESSION",
X_df_rt = demo_rt[cols_rt]
y       = demo_rt["y"]
strata  = demo_rt["strata"]
ids     = demo_rt["RELEASEID"].to_numpy()

In [ ]:
cat_col       = ['EDLEVEL09']
marital_col   = ['marital_st']
marital_cat   = [["Single", "Married", "Seperated/divorced/widowed"]]
preprocess_rt = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["DOC_AGE_HC3", "SHORT_EMSE", "rt_q1", "rt_q2","rt_q3"]),
        ("edu", imbPipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OrdinalEncoder(categories=[['No Ed', 'O Level', 'Degree', 'A Level']],
                                   handle_unknown="use_encoded_value", unknown_value=-1
                                  )),
        ]), cat_col),
        ("mar", imbPipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(categories=marital_cat,
                                 handle_unknown="ignore")),
            
        ]), marital_col),
    ],
    remainder="drop"
    # remainder="passthrough"
)

In [ ]:
# for loop for k-fold cross validation
results_all_rt = {}
feature_names = None
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model in models.items():
    print(f"\n=== Running Model: {model_name} ===")
    aucs, acc, auprcs, f1s, mccs, balanced_acc_rts = [], [], [], [], [], []
    y_true_all, y_prob_all, y_pred_all = [], [], []
    pr_per_fold, thr_rows = [], []
    per_fold_frames = []
    perm_imp = []

    memory = Memory(location=".cache_pipeline", verbose=0)
    pipeline_rt = imbPipeline(
        steps = [
            ("pre", preprocess_rt),
            ("ros", RandomOverSampler(sampling_strategy= "minority", random_state=42)),
            ("model", model)       
        ],
        memory=None, verbose=True,
    )
    
    for fold, (train_idx, test_idx) in enumerate(cv.split(X_df_rt, strata), 1):
        X_tr, X_te = X_df_rt.iloc[train_idx], X_df_rt.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        # grp_tr, grp_te = groups[train_idx], groups[test_idx]
        ids_test = ids[test_idx]

        # with oversampling
        pipeline_rt.fit(X_tr, y_tr)
        
        if feature_names is None:
            feature_names = pipeline_rt.named_steps["pre"].get_feature_names_out()
            
        y_prob = pipeline_rt.predict_proba(X_te)[:,1]
        y_pred = (y_prob >= 0.5).astype(int)

        # thr_rows = []
        for thr in [0.3, 0.4, 0.5, 0.6, 0.65, 0.7]:
            print(f"current fold {fold}")
            y_pred_thr = (y_prob >= thr).astype(int)
            if len(np.unique(y_pred_thr)) < 2:
                continue
            tn_t, fp_t, fn_t, tp_t = confusion_matrix(
                y_te, y_pred_thr).ravel()
            thr_rows.append({
                "fold":fold, "threshold":thr,
                "sensitivity": tp_t / (tp_t + fn_t + 1e-12),
                "specificity": tn_t / (tn_t + fp_t + 1e-12),
                "mcc": matthews_corrcoef(y_te, y_pred_thr),
            })
            
        aucs.append(roc_auc_score(y_te, y_prob))
        acc.append(accuracy_score(y_te, y_pred))
        auprc = average_precision_score(y_te, y_prob)
        balanced_acc_rt = balanced_accuracy_score(y_te, y_pred)
        mcc = matthews_corrcoef(y_te, y_pred)
        prec, rec, thr = precision_recall_curve(y_te, y_prob)
        f1 = f1_score(y_te, y_pred)
    
        # best f1 on this fold
        f1_best = 2*prec*rec / (prec+rec+ 1e-12)
        i_best = f1_best.argmax()
        best_thr = thr[i_best-1] if i_best > 0 else thr[0]
    
        pr_per_fold.append({"precision":prec, "recall":rec, "thresholds":thr, "fold":fold})
        print(f"Fold {fold}: AUPRC={auprc: .3f} | best F1={f1_best[i_best]} at thr={best_thr:.3}")
        
        fold_df = pd.DataFrame({
            "fold":fold,
            "id_col": ids_test,
            "y_true": y_te,
            "y_prob": y_prob,
            "y_pred": y_pred
        })
        auprcs.append(auprc)
        f1s.append(f1)
        mccs.append(mcc)
        balanced_acc_rts.append(balanced_acc_rt)
        per_fold_frames.append(fold_df)
        y_true_all.append(y_te)
        y_pred_all.append(y_pred)
        conf_mat = confusion_matrix(y_te, y_pred)
        print(conf_mat)
        tn, fp, fn, tp = conf_mat.ravel()
   
        # permutation importance
        perm = permutation_importance(
            pipeline_rt, X_te, y_te,
            # scoring="roc_auc",
            n_repeats=20,
            n_jobs=-1,
            random_state=42
            )
    
        y_prob_all.append(y_prob)
        perm_imp.append(perm.importances_mean)
        print(f"Fold {fold}: n]{len(test_idx)} | AUC={aucs[-1]:.3f} | Acc{acc[-1]:.3f}")
     # aggregate across folds   
    df_all_folds = pd.concat(per_fold_frames)
    results_all_rt[model_name] = {
        'mean_auc': np.mean(aucs),
        'mean_auprc': np.mean(auprcs),
        'mean_mcc': np.mean(mccs),
        'results_df': df_all_folds,
        'permutation_imp': perm_imp,
        'f1': np.mean(f1s),
        'thr_df'     : pd.DataFrame(thr_rows),
        'balanced_acc_rt_mean': np.mean(balanced_acc_rts),
        'balanced_acc_rt_st': np.std(balanced_acc_rts)
    }
# summary = pd.DataFrame({k:{"ROC_AUC":v['mean_auc'], ' AUPRC':v['mean_auprc'], 'f1':v['f1'], 'MCC':v['mean_mcc']} for k, v, in results_all.items()})
# print("\n=== Model Comparison ===")
# print(summary.round(3))

In [ ]:
summary = pd.DataFrame({k:{"ROC_AUC":v['mean_auc'], ' AUPRC':v['mean_auprc'], 'f1':np.mean(v['f1']), 'MCC':v['mean_mcc'] } for k, v, in results_all_rt.items()})
print("\n=== Model Comparison ===")
print(summary.round(3))

In [ ]:
with open("results_new/results_all_rt.pkl", "wb") as f:
    pickle.dump(results_all_rt, f)

## Save the results

In [ ]:
final_results = {}
for model_name in results_all_rt.keys():
    y_true = results_all_rt[model_name]["results_df"]["y_true"]
    y_pred = (results_all_rt[model_name]["results_df"]["y_prob"] >= 0.5).astype(int)
    # y_pred = results_all_rt[model_name]["results_df"]["y_pred"]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    # metrics
    specificity = tn /(tn + fp)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    pr_auc = average_precision_score(y_true, y_pred)
    balanced_acc_rt = balanced_accuracy_score(y_true, y_pred)
    balanc_rt_mean = results_all_rt[model_name]["balanced_acc_rt_mean"]
    balanc_rt_st = results_all_rt[model_name]["balanced_acc_rt_st"]
    

    final_results[model_name] = {
        "TN": tn,
        "FP":fp,
        "FN":fn,
        "TP":tp,
        "specificity":specificity,
        "precision":precision,
        "recall": recall,
        "mcc": mcc,
        "f1":f1,
        "brier":brier,
        "auc":auc,
        "pr_auc":pr_auc,
        "balanced_acc":balanced_acc_rt,
        "balance_mean": balanc_rt_mean,
        "balanc_cog_st": balanc_rt_st
    }

In [ ]:
summary_df = pd.DataFrame(final_results).T  # RTs 
print(summary_df)

In [ ]:
summary_df.to_csv("results_new/mlsum_rt.txt", sep=",", float_format="%.4f")

## prepare fold summary

In [ ]:
with open("datasets/results_all_rt.pkl", "rb") as f:
    results_all_rt = pickle.load(f)

In [ ]:
# Get summary metrics for each fold
# THRESHOLD = 0.5
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.65, 0.7]

summary_rows = []

for model_name in results_all_rt.keys():
    df = results_all_rt[model_name]["results_df"]
    for fold_id in sorted(df["fold"].unique()):
        fold_mask = df["fold"] == fold_id
        y_true = df.loc[fold_mask, "y_true"]
        y_prob = df.loc[fold_mask, "y_prob"]
        
       
        for thr in THRESHOLDS:
            y_pred = (y_prob >= thr).astype(int)
            
            if len(y_true.unique()) < 2:
                continue
                
            tn, fp, fn, tp = confusion_matrix(
                y_true, y_pred).ravel()
            sens = tp / (tp + fn + 1e-12)
            spec = tn / (tn + fp + 1e-12)
            bal_acc = (sens + spec) / 2

            summary_rows.append({
                "model": model_name,
                "fold": fold_id,
                "threshold":thr,
                "TN": tn,
                "FP": fp,
                "FN": fn,
                "TP": tp,
                "sensitivity": round(sens, 3),
                "specificity": round(spec, 3),
                "balanced_acc": round(bal_acc, 4),
                "mcc":          round(matthews_corrcoef(y_true, y_pred),4),
                "auc":         round(roc_auc_score(y_true, y_prob), 4),
                "pr_auc":      round(average_precision_score(y_true, y_prob), 4),
                "brier":       round(brier_score_loss(y_true, y_prob), 4)
            })

fold_summary = pd.DataFrame(summary_rows)
print(fold_summary)

agg_summary = (fold_summary
              .groupby("model")[["sensitivity","specificity", "balanced_acc", "mcc", "auc", "pr_auc", "brier"]].agg(["mean", "std"]).round(4))

fold_summary.to_csv("results_new/fold_sum_rtmodel.csv", index=False)

print("saved fold summary")
print(fold_summary.head())

In [ ]:
# figure for sensitivty analyses to justify the threshold
fig, axes = plt.subplots(1, 2, figsize=(12,5))
colors={"Logistic Regression": "#000000",
       "RBF SVM": "#666666",
        # "Random Forest": "#2ca02c",
        # "Gaussian NB": "#d62728",
        # "XgBoost": "#9467bd"
       }

for metric, ax, ylabel in zip(["sensitivity", "balanced_acc"], axes, ["Sensitivity", "Balanced acc"]):
    for model_name, color in colors.items():
        subset = fold_summary[fold_summary["model"]==model_name]

        grouped = (subset.groupby("threshold")[metric]
            .agg(["mean", "std"]).reset_index())
        ax.plot(grouped["threshold"], grouped["mean"], 
               label=model_name, color=color, marker="o")
        ax.fill_between(
            grouped["threshold"],
            grouped["mean"] - grouped["std"],
            grouped["mean"] + grouped["std"],
            alpha=0.5, color=color)
    ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Reported threshold (0.5)")
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()